# 01 · ETL — Desertos de Saúde (SP)
**Objetivo:** montar a base de municípios de SP (geometria, população, PIB) e a base de estabelecimentos de saúde (CNES).

**Saídas (Drive → `Trabalho UBD/etl_output`):** `municipios-sp.geojson` e `cnes-sp.geojson`, em EPSG:31983.

**Como rodar:** Ambiente de execução → Executar tudo (as células seguem a ordem).

## 1. Preparação — bibliotecas e pasta do Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import requests
import pandas as pd
import geopandas as gpd
import os

BASE = "/content/drive/MyDrive/Trabalho UBD"   # pasta do trabalho no Drive
SAIDA = f"{BASE}/etl_output"
os.makedirs(SAIDA, exist_ok=True)

## 2. Parte A — Municípios de SP (malha do IBGE)

In [ ]:
# Baixa o mapa dos 645 municípios (geometria)
r = requests.get(
    "https://servicodados.ibge.gov.br/api/v2/malhas/35",
    params={"resolucao": 5, "formato": "application/vnd.geo+json", "qualidade": 4},
    timeout=120,
)
r.raise_for_status()
mun = gpd.GeoDataFrame.from_features(r.json()["features"], crs="EPSG:4326")
mun["cod_ibge"] = mun["codarea"].astype(str)   # código de 7 dígitos
print(len(mun), "municípios")

## 3. Parte A — Nomes dos municípios (a malha não traz o nome)

In [ ]:
lista = requests.get(
    "https://servicodados.ibge.gov.br/api/v1/localidades/estados/35/municipios", timeout=60
).json()
nomes = pd.DataFrame({"cod_ibge": [str(m["id"]) for m in lista],
                      "municipio": [m["nome"] for m in lista]})
mun = mun.merge(nomes, on="cod_ibge", how="left")
print(mun["municipio"].isna().sum(), "municípios sem nome (esperado: 0)")

## 4. Parte A — População (Censo 2022) e PIB (SIDRA/IBGE)

In [ ]:
def sidra(url, nome_valor):
    """Lê uma tabela do SIDRA e devolve código do município + valor."""
    j = requests.get(url, timeout=120).json()
    df = pd.DataFrame(j[1:])                        # a linha 0 é o cabeçalho
    df = df.rename(columns={"D1C": "cod_ibge", "V": nome_valor})
    df[nome_valor] = pd.to_numeric(df[nome_valor], errors="coerce")
    return df[["cod_ibge", nome_valor]]

pop = sidra("https://apisidra.ibge.gov.br/values/t/4714/n6/in%20n3%2035/v/93/p/2022", "populacao")
pib = sidra("https://apisidra.ibge.gov.br/values/t/5938/n6/in%20n3%2035/v/37/p/last", "pib_mil_reais")
mun = mun.merge(pop, on="cod_ibge", how="left").merge(pib, on="cod_ibge", how="left")
print(mun[["populacao", "pib_mil_reais"]].describe())

## 5. Parte A — Reprojeção para EPSG:31983 (metros) e área

In [ ]:
mun = mun.to_crs(epsg=31983)                        # sistema em metros
mun["area_km2"] = mun.geometry.area / 1e6
mun["cod6"] = mun["cod_ibge"].str[:6]               # chave para juntar com o CNES
print(mun.crs)

## 6. Parte B — Leitos de internação (CNES/LT, FTP do DATASUS)
Baixa o arquivo de leitos de SP, soma os leitos existentes por município e compara com a rodada anterior (92.265 leitos de internação).

In [ ]:
%pip -q install datasus-dbc dbfread

In [ ]:
import ftplib, datasus_dbc
from dbfread import DBF

COMPETENCIA = "2608"                       # AAMM do arquivo (2608 = agosto/2026)
FTP_HOST = "ftp.datasus.gov.br"

def baixar_dbc(grupo, uf, aamm):
    """Baixa GRUPO+UF+AAMM.dbc do FTP do DATASUS (ex.: LTSP2608.dbc)."""
    nome = f"{grupo}{uf}{aamm}.dbc"
    destino = f"/content/{nome}"
    with ftplib.FTP(FTP_HOST, timeout=120) as ftp:
        ftp.login()
        ftp.cwd(f"/dissemin/publicos/CNES/200508_/Dados/{grupo}")
        arquivos = ftp.nlst()
        if nome not in arquivos:
            ultimos = sorted(a for a in arquivos if a.startswith(f"{grupo}{uf}"))[-3:]
            raise FileNotFoundError(f"{nome} não existe no FTP. Últimos disponíveis: {ultimos}")
        with open(destino, "wb") as f:
            ftp.retrbinary(f"RETR {nome}", f.write)
    return destino

def ler_dbc(caminho):
    """Converte .dbc em .dbf e lê como tabela."""
    dbf = caminho.replace(".dbc", ".dbf")
    datasus_dbc.decompress(caminho, dbf)
    return pd.DataFrame(list(DBF(dbf, encoding="latin-1")))

lt = ler_dbc(baixar_dbc("LT", "SP", COMPETENCIA))
print(len(lt), "linhas | colunas:", list(lt.columns))

In [ ]:
# Quantidade de leitos existentes por tipo de leito
lt["QT_EXIST"] = pd.to_numeric(lt["QT_EXIST"], errors="coerce").fillna(0)
lt["TP_LEITO"] = lt["TP_LEITO"].astype(str).str.strip().str.zfill(2)
# 01 cirúrgico, 02 clínico, 03 complementar (UTI etc.), 04 obstétrico, 05 pediátrico, 06 outras especialidades, 07 hospital-dia
print(lt.groupby("TP_LEITO")["QT_EXIST"].sum().astype(int))

# Compara alguns critérios com a rodada anterior (92.265 leitos de internação)
criterios = {"01,02,04,05,06": ["01", "02", "04", "05", "06"],
             "+ hospital-dia (07)": ["01", "02", "04", "05", "06", "07"],
             "+ complementares (03)": ["01", "02", "03", "04", "05", "06"],
             "todos": ["01", "02", "03", "04", "05", "06", "07"]}
for nome, tipos in criterios.items():
    print(f"{nome:24s} {int(lt[lt['TP_LEITO'].isin(tipos)]['QT_EXIST'].sum()):>8,}")

In [ ]:
# Critério escolhido: use o que mais se aproxima de 92.265 na tabela acima
TIPOS_INTERNACAO = ["01", "02", "04", "05", "06"]

lt_int = lt[lt["TP_LEITO"].isin(TIPOS_INTERNACAO)].copy()
lt_int["cod6"] = lt_int["CODUFMUN"].astype(str).str[:6]
leitos = lt_int.groupby("cod6")["QT_EXIST"].sum().astype(int).rename("leitos").reset_index()

mun = mun.drop(columns=["leitos", "leitos_10k"], errors="ignore").merge(leitos, on="cod6", how="left")
mun["leitos"] = mun["leitos"].fillna(0).astype(int)
mun["leitos_10k"] = mun["leitos"] / mun["populacao"] * 10_000

print("Leitos de internação em SP:", f"{mun['leitos'].sum():,}", "(rodada anterior: 92.265)")
print("Municípios sem leito:", int((mun["leitos"] == 0).sum()), "(rodada anterior: 293)")

## 7. Parte B — Estabelecimentos com coordenadas (CNES, dados abertos do SUS)
Baixa o cadastro de estabelecimentos, fica só com SP, classifica cada um (hospital, urgência etc.) e cria os pontos.
O código procura sozinho as colunas de latitude, longitude, município e tipo. Ele mostra o que achou; se errar, preencha os nomes na primeira linha da célula.

In [ ]:
import re, json, zipfile

URL_CNES = "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/CNES/cnes_estabelecimentos_json.zip"
COL_LAT = COL_LON = COL_MUN = COL_TIPO = None     # preencha só se a detecção automática falhar

# Baixa e abre o arquivo
resp = requests.get(URL_CNES, stream=True, timeout=600)
resp.raise_for_status()
with open("/content/cnes.zip", "wb") as f:
    for parte in resp.iter_content(1 << 20):
        f.write(parte)
with zipfile.ZipFile("/content/cnes.zip") as z:
    arquivo = z.namelist()[0]
    z.extract(arquivo, "/content")

# Lê o JSON (lista de registros, ou um objeto que contém a lista, ou uma linha por registro)
caminho = f"/content/{arquivo}"
try:
    with open(caminho, encoding="utf-8") as f:
        dados = json.load(f)
    if isinstance(dados, dict):
        dados = next(v for v in dados.values() if isinstance(v, list))
    est = pd.DataFrame(dados)
    del dados
except json.JSONDecodeError:
    est = pd.read_json(caminho, lines=True)
print(len(est), "estabelecimentos no Brasil | colunas:", list(est.columns))

def achar(df, *padroes):
    """Primeira coluna cujo nome combina com algum dos padrões."""
    for p in padroes:
        for c in df.columns:
            if re.search(p, str(c).lower()):
                return c
    return None

COL_LAT = COL_LAT or achar(est, r"latitude", r"^lat")
COL_LON = COL_LON or achar(est, r"longitude", r"^lon")
COL_MUN = COL_MUN or achar(est, r"^co_ibge$", r"codigo_municipio", r"co_municipio", r"codigo_ibge", r"cod.*munic")
COL_TIPO = COL_TIPO or achar(est, r"codigo_tipo_unidade", r"tp_unid", r"tipo_unidade", r"tipo.*(unidade|estab)")
print("Colunas usadas:", {"lat": COL_LAT, "lon": COL_LON, "município": COL_MUN, "tipo": COL_TIPO})
assert None not in (COL_LAT, COL_LON, COL_MUN, COL_TIPO), "Não achei alguma coluna: preencha COL_LAT, COL_LON, COL_MUN e COL_TIPO."

In [ ]:
# Só São Paulo (código do município começa com 35)
est["cod6_cnes"] = est[COL_MUN].astype(str).str.replace(r"\.0$", "", regex=True).str[:6]
est = est[est["cod6_cnes"].str.startswith("35")].copy()

# Categoria do estabelecimento (pelo código do tipo de unidade, ou pelo texto)
CODIGOS = {"05": "Hospital", "07": "Hospital", "73": "UPA / Pronto atendimento",
           "20": "Pronto-socorro", "21": "Pronto-socorro", "15": "Unidade mista",
           "01": "Atenção básica", "02": "Atenção básica"}

def categoria(valor):
    v = str(valor).strip().lower().replace(".0", "")
    if v.isdigit():
        return CODIGOS.get(v.zfill(2), "Outros")
    if "pronto" in v and "socorro" in v:
        return "Pronto-socorro"
    if "upa" in v or "pronto atend" in v:
        return "UPA / Pronto atendimento"
    if "mista" in v:
        return "Unidade mista"
    if "hospital" in v and "dia" not in v:
        return "Hospital"
    if "centro de sa" in v or "posto de sa" in v or "básica" in v or "basica" in v:
        return "Atenção básica"
    return "Outros"

est["categoria"] = est[COL_TIPO].map(categoria)

# Coordenadas: número, dentro da caixa aproximada de SP (descarta zeros e inversões)
est["lat"] = pd.to_numeric(est[COL_LAT].astype(str).str.replace(",", "."), errors="coerce")
est["lon"] = pd.to_numeric(est[COL_LON].astype(str).str.replace(",", "."), errors="coerce")
antes = len(est)
est = est[est["lat"].between(-25.5, -19.5) & est["lon"].between(-53.5, -44.0)]
print(f"{len(est)} de {antes} estabelecimentos de SP com coordenada válida")

# Pontos em EPSG:31983 e município onde o ponto cai de fato (cod6_geo)
pts = gpd.GeoDataFrame(est[["cod6_cnes", "categoria"]].reset_index(drop=True),
                       geometry=gpd.points_from_xy(est["lon"], est["lat"]), crs="EPSG:4326").to_crs(epsg=31983)
pts = gpd.sjoin(pts, mun[["cod6", "geometry"]], how="left", predicate="within").drop(columns="index_right")
pts = pts.rename(columns={"cod6": "cod6_geo"})
print("Pontos fora de qualquer município:", int(pts["cod6_geo"].isna().sum()))
print(pts["categoria"].value_counts())

In [ ]:
# Indicadores por município (contagem pelo município do cadastro)
cont = pts.groupby(["cod6_cnes", "categoria"]).size().unstack(fill_value=0)
for col in ["Hospital", "UPA / Pronto atendimento", "Pronto-socorro"]:
    if col not in cont.columns:
        cont[col] = 0
cont["n_hospitais"] = cont["Hospital"]
cont["n_urgencia"] = cont["UPA / Pronto atendimento"] + cont["Pronto-socorro"]
cont = cont[["n_hospitais", "n_urgencia"]].reset_index().rename(columns={"cod6_cnes": "cod6"})

mun = mun.drop(columns=["n_hospitais", "n_urgencia", "deserto_urgencia"], errors="ignore").merge(cont, on="cod6", how="left")
mun[["n_hospitais", "n_urgencia"]] = mun[["n_hospitais", "n_urgencia"]].fillna(0).astype(int)
mun["deserto_urgencia"] = mun["n_urgencia"] == 0          # sem pronto-socorro nem UPA

# Conferência com a rodada anterior
print("Estabelecimentos:", f"{len(pts):,}", "(rodada anterior: 115.754)")
print("Municípios sem UPA/pronto-socorro:", int(mun["deserto_urgencia"].sum()), "(rodada anterior: 379)")
print("Municípios sem leito:", int((mun["leitos"] == 0).sum()), "(rodada anterior: 293)")

## 8. Exportar para o Drive (com verificação)

In [ ]:
# Confere se a Parte B foi executada antes de salvar
colunas_mun = ["municipio", "populacao", "pib_mil_reais", "area_km2", "cod6", "leitos",
               "leitos_10k", "n_urgencia", "deserto_urgencia", "n_hospitais"]
colunas_pts = ["categoria", "cod6_geo"]

assert "pts" in globals(), "Rode a Parte B (seções 6 e 7) antes de exportar."
falta = [c for c in colunas_mun if c not in mun.columns] + [c for c in colunas_pts if c not in pts.columns]
assert not falta, f"Colunas ausentes: {falta}"

mun.to_file(f"{SAIDA}/municipios-sp.geojson", driver="GeoJSON")
pts.to_file(f"{SAIDA}/cnes-sp.geojson", driver="GeoJSON")
print(f"Salvo em {SAIDA}: {len(mun)} municípios e {len(pts)} estabelecimentos")
print("Leitos de internação no estado:", int(mun["leitos"].sum()))